# Cross-Sell Prediction with Temporal Modeling
**Portfolio / GitHub Version with Realistic Synthetic Data**

This notebook presents an end-to-end cross-sell prediction pipeline using **customer-level temporal data**.

Because the original technical case dataset cannot be publicly shared, this notebook includes a **realistic synthetic data generator** designed to preserve key properties of real-world customer behavior:

- customer heterogeneity
- temporal autocorrelation
- non-random product adoption
- behavioral acceleration before adoption
- censored future windows

The goal is to answer three business questions:

1. **Who** should be approached for cross-sell?
2. **When** is the best moment to approach them?
3. **Why** are some customers more likely to adopt?

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve

from lightgbm import LGBMClassifier

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

## 2. Synthetic Data Generation

The synthetic dataset below was not created using independent random draws only.
Instead, it simulates a plausible business environment where:

- each customer has a latent maturity profile
- product usage evolves over time with persistence
- customers differ in usage intensity and system breadth
- cross-sell adoption depends on historical behavior
- adoption becomes more likely after progressive engagement growth

This makes the dataset much more realistic for portfolio purposes.

In [ ]:
def generate_synthetic_cross_sell_data(
    n_customers=2500,
    start_date="2023-01-01",
    n_months=24,
    seed=42
):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start=start_date, periods=n_months, freq="MS")

    rows = []

    segments = np.array(["small", "mid", "large"])
    segment_probs = np.array([0.50, 0.35, 0.15])

    for cust_id in range(1, n_customers + 1):
        segment = rng.choice(segments, p=segment_probs)

        segment_scale = {
            "small": 0.8,
            "mid": 1.1,
            "large": 1.5
        }[segment]

        maturity = np.clip(rng.normal(0.0, 1.0), -2.5, 2.5)
        cross_sell_affinity = np.clip(rng.normal(0.0, 1.0), -2.5, 2.5)
        volatility_profile = np.clip(rng.normal(0.0, 0.7), -1.5, 1.5)
        growth_profile = np.clip(rng.normal(0.08, 0.10), -0.15, 0.35)

        observed_months = int(rng.integers(low=max(12, n_months - 8), high=n_months + 1))
        customer_dates = dates[:observed_months]

        base_usage = max(2, rng.lognormal(mean=2.2 + 0.20 * maturity, sigma=0.45) * segment_scale)

        adopted_b = 0
        adoption_month = None

        prev_total = base_usage * rng.uniform(0.85, 1.15)
        prev_active_systems = max(1, int(round(rng.normal(2 + 0.8 * segment_scale + 0.5 * maturity, 1))))

        for t, date in enumerate(customer_dates):
            life_time_m = t + 1
            seasonal = 1 + 0.08 * np.sin(2 * np.pi * (date.month / 12))
            persistence = 0.72
            trend_multiplier = 1 + growth_profile * np.log1p(life_time_m) / 3.5
            random_shock = rng.normal(0, 0.10 + 0.03 * abs(volatility_profile))

            current_total = (
                persistence * prev_total
                + (1 - persistence) * base_usage * trend_multiplier * seasonal
            ) * (1 + random_shock)

            current_total = max(0.5, current_total)

            # number of active systems grows with maturity and usage
            expected_systems = (
                1.2
                + 0.55 * np.log1p(current_total)
                + 0.55 * maturity
                + 0.35 * segment_scale
                + 0.03 * life_time_m
            )
            qtd_sistemas_ativos = int(np.clip(np.round(rng.normal(expected_systems, 0.8)), 1, 6))

            # split total usage across 6 systems
            system_weights = rng.dirichlet(alpha=np.ones(6) * (0.7 + 0.2 * segment_scale))
            raw_usage = current_total * system_weights

            active_mask = np.array([1 if i < qtd_sistemas_ativos else 0 for i in range(6)])
            rng.shuffle(active_mask)
            system_usage = raw_usage * active_mask

            # introduce realistic noise and sparse systems
            system_usage = np.where(system_usage < 0.15, 0, system_usage)
            uso_sistema_1, uso_sistema_2, uso_sistema_3, uso_sistema_4, uso_sistema_5, uso_sistema_6 = system_usage

            produto_a = 1  # base product already present in the portfolio framing

            # behavioral signals that influence adoption
            usage_growth = (current_total - prev_total) / max(prev_total, 1e-6)
            breadth_signal = qtd_sistemas_ativos / 6
            maturity_signal = np.tanh(maturity / 1.5)
            engagement_signal = np.tanh(np.log1p(current_total) / 2.5)

            if adopted_b == 0:
                logit_p = (
                    -4.1
                    + 0.90 * engagement_signal
                    + 1.05 * breadth_signal
                    + 0.40 * np.tanh(life_time_m / 12)
                    + 0.55 * np.clip(usage_growth, -1, 1)
                    + 0.55 * maturity_signal
                    + 0.45 * np.tanh(cross_sell_affinity / 1.5)
                    + 0.10 * segment_scale
                )
                p_adopt = 1 / (1 + np.exp(-logit_p))
                adopted_now = rng.binomial(1, p_adopt)
                if adopted_now == 1:
                    adopted_b = 1
                    adoption_month = date
            produto_b = adopted_b

            # after adoption, usage typically expands
            if adopted_b == 1:
                current_total = current_total * rng.uniform(1.03, 1.10)

            # realistic missingness in a few observations
            maybe_missing = rng.random()
            if maybe_missing < 0.01:
                uso_sistema_5 = np.nan
            if maybe_missing > 0.98:
                uso_sistema_6 = np.nan

            rows.append({
                "ID": cust_id,
                "data": date,
                "segmento": segment,
                "maturidade_latente": maturity,
                "afinidade_cross_sell": cross_sell_affinity,
                "life_time_m": life_time_m,
                "produto_a": produto_a,
                "produto_b": produto_b,
                "uso_sistema_1": uso_sistema_1,
                "uso_sistema_2": uso_sistema_2,
                "uso_sistema_3": uso_sistema_3,
                "uso_sistema_4": uso_sistema_4,
                "uso_sistema_5": uso_sistema_5,
                "uso_sistema_6": uso_sistema_6,
            })

            prev_total = current_total
            prev_active_systems = qtd_sistemas_ativos

    df = pd.DataFrame(rows).sort_values(["ID", "data"]).reset_index(drop=True)

    # observed usage aggregates
    uso_cols = [c for c in df.columns if c.startswith("uso_sistema_")]
    df["uso_total"] = df[uso_cols].fillna(0).sum(axis=1)
    df["qtd_sistemas_ativos"] = (df[uso_cols].fillna(0) > 0).sum(axis=1)

    # first adoption date
    first_adoption = (
        df.loc[df["produto_b"] == 1]
        .groupby("ID", as_index=False)["data"]
        .min()
        .rename(columns={"data": "primeira_data_b"})
    )

    df = df.merge(first_adoption, on="ID", how="left")
    return df

df = generate_synthetic_cross_sell_data(n_customers=2500, n_months=24, seed=SEED)
df.head()

,ID,data,segmento,maturidade_latente,afinidade_cross_sell,life_time_m,produto_a,produto_b,uso_sistema_1,uso_sistema_2,uso_sistema_3,uso_sistema_4,uso_sistema_5,uso_sistema_6,uso_total,qtd_sistemas_ativos,primeira_data_b
0,1,2023-01-01,mid,-1.039984,0.750451,1,1,0,3.655733,2.336214,0.298735,0.0,0.000000,0.000000,6.290682,3,2023-05-01
1,1,2023-02-01,mid,-1.039984,0.750451,2,1,0,1.382918,4.525396,0.000000,0.0,0.000000,1.329649,7.237963,3,2023-05-01
2,1,2023-03-01,mid,-1.039984,0.750451,3,1,0,0.312812,0.000000,2.640418,0.0,0.000000,0.000000,2.953230,2,2023-05-01
3,1,2023-04-01,mid,-1.039984,0.750451,4,1,0,1.249442,1.135049,0.000000,0.0,0.000000,0.000000,2.384492,2,2023-05-01
4,1,2023-05-01,mid,-1.039984,0.750451,5,1,1,0.000000,0.669215,0.000000,0.0,1.955275,0.500159,3.124649,3,2023-05-01


## 3. Audit and Initial Understanding

This section validates whether the synthetic dataset has realistic scale, coverage, and event behavior.

In [ ]:
print("Shape:", df.shape)
print("Customers:", df["ID"].nunique())
print("Date range:", df["data"].min(), "to", df["data"].max())
print("Duplicates:", df.duplicated().sum())

missing = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing[missing > 0].head(20))

adoption_rate_customer = (
    df.groupby("ID")["produto_b"].max().mean()
)
print(f"Customer-level adoption rate of Product B: {adoption_rate_customer:.2%}")

Shape: (49875, 17)
Customers: 2500
Date range: 2023-01-01 00:00:00 to 2024-12-01 00:00:00
Duplicates: 0


,0
primeira_data_b,24.038095
uso_sistema_6,1.992982
uso_sistema_5,1.056642


Customer-level adoption rate of Product B: 75.40%


## 4. Exploratory Data Analysis (EDA)

The EDA aims to answer:

1. Does the dataset have stable temporal coverage?
2. Is Product B adoption concentrated in specific periods?
3. Do adopters differ from non-adopters?
4. Are there visible pre-adoption behavioral signals?

In [ ]:
base_mensal = (
    df.groupby("data")
      .agg(
          clientes_unicos=("ID", "nunique"),
          registros=("ID", "size"),
          adotantes_b=("produto_b", "sum")
      )
      .reset_index()
)

fig = px.line(base_mensal, x="data", y="clientes_unicos", title="Unique customers observed by month")
fig.show()

fig = px.line(base_mensal, x="data", y="adotantes_b", title="Customers with Product B by month")
fig.show()

In [ ]:
adocao_cliente = (
    df.groupby("ID")["produto_b"]
      .max()
      .rename("ja_adotou_b")
      .reset_index()
)

foto_cliente = (
    df.groupby("ID")
      .agg(
          uso_total_medio=("uso_total", "mean"),
          uso_total_mediano=("uso_total", "median"),
          qtd_sistemas_media=("qtd_sistemas_ativos", "mean"),
          qtd_sistemas_max=("qtd_sistemas_ativos", "max"),
          life_time_max=("life_time_m", "max")
      )
      .reset_index()
      .merge(adocao_cliente, on="ID", how="left")
)

display(
    foto_cliente.groupby("ja_adotou_b")[["uso_total_medio", "qtd_sistemas_media", "life_time_max"]]
    .agg(["mean", "median"])
)

fig = px.box(foto_cliente, x="ja_adotou_b", y="uso_total_medio",
             title="Average total usage: adopters vs non-adopters")
fig.show()

fig = px.box(foto_cliente, x="ja_adotou_b", y="qtd_sistemas_media",
             title="Average active systems: adopters vs non-adopters")
fig.show()

uso_total_medio           qtd_sistemas_media        life_time_max  \
                       mean    median               mean median          mean   
ja_adotou_b                                                                     
0                  4.352912  3.516967           2.449465   2.45     19.494309   
1                  7.407694  5.777265           3.005852   3.00     20.098674   

                    
            median  
ja_adotou_b         
0             19.0  
1             20.0

In [ ]:
df["adotou_agora"] = (
    (df["produto_b"] == 1) &
    (df.groupby("ID")["produto_b"].shift(1).fillna(0) == 0)
).astype(int)

primeira_adocao = (
    df.loc[df["adotou_agora"] == 1, ["ID", "data"]]
      .sort_values(["ID", "data"])
      .groupby("ID", as_index=False)
      .first()
      .rename(columns={"data": "data_adocao"})
)

janela = df.merge(primeira_adocao, on="ID", how="inner")

janela["meses_para_adocao"] = (
    (janela["data_adocao"].dt.year - janela["data"].dt.year) * 12 +
    (janela["data_adocao"].dt.month - janela["data"].dt.month)
)

janela_pre = janela[
    (janela["meses_para_adocao"] >= 0) &
    (janela["meses_para_adocao"] <= 6)
].copy()

trajetoria = (
    janela_pre.groupby("meses_para_adocao")
    .agg(
        uso_total_medio=("uso_total", "mean"),
        sistemas_ativos_medios=("qtd_sistemas_ativos", "mean")
    )
    .reset_index()
    .sort_values("meses_para_adocao", ascending=False)
)

fig = px.line(trajetoria, x="meses_para_adocao", y="uso_total_medio",
              title="Average total usage before adoption")
fig.show()

fig = px.line(trajetoria, x="meses_para_adocao", y="sistemas_ativos_medios",
              title="Average active systems before adoption")
fig.show()

display(trajetoria)

,meses_para_adocao,uso_total_medio,sistemas_ativos_medios
6,6,5.728603,2.667671
5,5,5.836664,2.764862
4,4,6.016230,2.742994
3,3,6.256927,2.831762
2,2,6.318907,2.822094
1,1,6.247340,2.806358
0,0,6.729558,2.957560


### Executive Readout from EDA

The synthetic data reproduces a realistic pattern:

- adoption is not random
- adopters tend to be more mature and engaged
- breadth of system usage matters
- engagement tends to accelerate before adoption

These signals support the use of temporal predictive modeling.

## 5. Problem Formulation

We formulate the challenge as a **temporal supervised learning** problem.

For each **customer-month**, the goal is to estimate:

> **Will this customer adopt Product B within the next 3 months?**

This requires:

- building a forward-looking target
- using only information available up to the reference month
- excluding post-adoption observations
- using chronological validation

In [ ]:
def criar_target_futuro(
    base: pd.DataFrame,
    horizonte: int = 3,
    id_col: str = "ID",
    date_col: str = "data",
    target_col: str = "produto_b"
) -> pd.DataFrame:
    base = base.sort_values([id_col, date_col]).copy()
    target_final = []

    for _, g in base.groupby(id_col):
        y = g[target_col].values
        target = np.full(len(g), np.nan)

        for i in range(len(g)):
            inicio = i + 1
            fim = i + 1 + horizonte

            if fim <= len(g):
                target[i] = int(np.max(y[inicio:fim]) > 0)

        target_final.extend(target)

    base[f"target_{horizonte}m"] = target_final
    return base

df = criar_target_futuro(df, horizonte=3)

# keep only pre-adoption observations
df_model = df[
    (df["primeira_data_b"].isna()) | (df["data"] < df["primeira_data_b"])
].copy()

# remove records without full future horizon
df_model = df_model.dropna(subset=["target_3m"]).copy()

print("Modeling base shape:", df_model.shape)
print("Target rate:", df_model["target_3m"].mean())

Modeling base shape: (23646, 19)
Target rate: 0.19123741859088217


## 6. Feature Engineering

Temporal feature engineering is a major source of predictive power in this problem.

We add:

- lag features
- month-over-month changes
- rolling averages and volatility
- short-term vs medium-term trend signals

In [ ]:
def criar_features_temporais(
    base: pd.DataFrame,
    cols: list,
    id_col: str = "ID",
    date_col: str = "data",
    windows: tuple = (3, 6)
) -> pd.DataFrame:
    base = base.sort_values([id_col, date_col]).copy()

    for col in cols:
        grp = base.groupby(id_col)[col]

        base[f"{col}_lag1"] = grp.shift(1)
        base[f"{col}_diff1"] = grp.diff(1)
        base[f"{col}_pct_change1"] = grp.pct_change(1).replace([np.inf, -np.inf], np.nan)

        shifted = grp.shift(1)

        for w in windows:
            roll = shifted.rolling(w, min_periods=1)
            base[f"{col}_rollmean_{w}"] = roll.mean().reset_index(level=0, drop=True)
            base[f"{col}_rollstd_{w}"] = roll.std().reset_index(level=0, drop=True)
            base[f"{col}_rollmin_{w}"] = roll.min().reset_index(level=0, drop=True)
            base[f"{col}_rollmax_{w}"] = roll.max().reset_index(level=0, drop=True)

        if f"{col}_rollmean_3" in base.columns and f"{col}_rollmean_6" in base.columns:
            base[f"{col}_trend_curto_vs_medio"] = (
                base[f"{col}_rollmean_3"] - base[f"{col}_rollmean_6"]
            )

    return base

feature_cols_temporais = [
    "uso_total",
    "qtd_sistemas_ativos",
    "uso_sistema_1",
    "uso_sistema_2",
    "uso_sistema_3",
    "uso_sistema_4",
    "uso_sistema_5",
    "uso_sistema_6",
]

df_model = criar_features_temporais(df_model, feature_cols_temporais)
df_model.head()

,ID,data,segmento,maturidade_latente,afinidade_cross_sell,life_time_m,produto_a,produto_b,uso_sistema_1,uso_sistema_2,...,uso_sistema_6_pct_change1,uso_sistema_6_rollmean_3,uso_sistema_6_rollstd_3,uso_sistema_6_rollmin_3,uso_sistema_6_rollmax_3,uso_sistema_6_rollmean_6,uso_sistema_6_rollstd_6,uso_sistema_6_rollmin_6,uso_sistema_6_rollmax_6,uso_sistema_6_trend_curto_vs_medio
0,1,2023-01-01,mid,-1.039984,0.750451,1,1,0,3.655733,2.336214,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2023-02-01,mid,-1.039984,0.750451,2,1,0,1.382918,4.525396,...,NaN,0.000000,NaN,0.0,0.000000,0.000000,NaN,0.0,0.000000,0.0
2,1,2023-03-01,mid,-1.039984,0.750451,3,1,0,0.312812,0.000000,...,-1.0,0.664824,0.940204,0.0,1.329649,0.664824,0.940204,0.0,1.329649,0.0
3,1,2023-04-01,mid,-1.039984,0.750451,4,1,0,1.249442,1.135049,...,NaN,0.443216,0.767673,0.0,1.329649,0.443216,0.767673,0.0,1.329649,0.0
20,2,2023-01-01,small,-0.794136,0.439637,1,1,0,0.000000,0.160203,...,NaN,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0


## 7. Temporal Split

To simulate production use, the data is split chronologically into:

- **train**
- **validation**
- **test**

This is critical to avoid optimistic bias.

In [ ]:
all_months = sorted(df_model["data"].unique())

train_end = all_months[int(len(all_months) * 0.60)]
valid_end = all_months[int(len(all_months) * 0.80)]

train = df_model[df_model["data"] <= train_end].copy()
valid = df_model[(df_model["data"] > train_end) & (df_model["data"] <= valid_end)].copy()
test  = df_model[df_model["data"] > valid_end].copy()

TARGET = "target_3m"
EXCLUIR = ["ID", "data", "produto_b", "primeira_data_b", "adotou_agora", TARGET]

features = [c for c in df_model.columns if c not in EXCLUIR]

X_train, y_train = train[features], train[TARGET]
X_valid, y_valid = valid[features], valid[TARGET]
X_test, y_test = test[features], test[TARGET]

print("Train:", X_train.shape, y_train.mean())
print("Valid:", X_valid.shape, y_valid.mean())
print("Test :", X_test.shape, y_test.mean())

Train: (20491, 109) 0.1896930359670099
Valid: (2451, 109) 0.20359037127702978
Test : (704, 109) 0.19318181818181818


## 8. Preprocessing

In [ ]:
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features)
    ]
)

## 9. Baseline Model — Logistic Regression

A transparent baseline helps establish whether more complex models are actually adding value.

In [ ]:
baseline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

baseline.fit(X_train, y_train)

p_valid_base = baseline.predict_proba(X_valid)[:, 1]
p_test_base = baseline.predict_proba(X_test)[:, 1]

print("BASELINE VALID ROC AUC:", roc_auc_score(y_valid, p_valid_base))
print("BASELINE VALID PR AUC :", average_precision_score(y_valid, p_valid_base))
print("BASELINE TEST ROC AUC :", roc_auc_score(y_test, p_test_base))
print("BASELINE TEST PR AUC  :", average_precision_score(y_test, p_test_base))

BASELINE VALID ROC AUC: 0.638084570780906
BASELINE VALID PR AUC : 0.3306639573767032
BASELINE TEST ROC AUC : 0.6452464788732395
BASELINE TEST PR AUC  : 0.3327271851920207


## 10. Main Model — LightGBM

In [ ]:
lgb_default = Pipeline([
    ("preprocess", preprocess),
    ("model", LGBMClassifier(
        class_weight="balanced",
        random_state=SEED
    ))
])

lgb_default.fit(X_train, y_train)

p_valid_lgb = lgb_default.predict_proba(X_valid)[:, 1]
p_test_lgb = lgb_default.predict_proba(X_test)[:, 1]

print("LGB VALID ROC AUC:", roc_auc_score(y_valid, p_valid_lgb))
print("LGB VALID PR AUC :", average_precision_score(y_valid, p_valid_lgb))
print("LGB TEST ROC AUC :", roc_auc_score(y_test, p_test_lgb))
print("LGB TEST PR AUC  :", average_precision_score(y_test, p_test_lgb))

[LightGBM] [Info] Number of positive: 3887, number of negative: 16604
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015712 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23216
[LightGBM] [Info] Number of data points in the train set: 20491, number of used features: 110
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LGB VALID ROC AUC: 0.6184089490456324
LGB VALID PR AUC : 0.32145281787788577
LGB TEST ROC AUC : 0.608611226180613
LGB TEST PR AUC  : 0.2584399434369231


## 11. Hyperparameter Tuning

Hyperparameters are optimized using **temporal validation** and **PR-AUC** as the primary objective.
This is especially relevant when the positive class is relatively sparse.

In [ ]:
param_grid = {
    "n_estimators": [200, 400],
    "learning_rate": [0.01, 0.03, 0.05],
    "num_leaves": [15, 31, 63],
    "min_child_samples": [20, 50],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
    "reg_alpha": [0.0, 0.5],
    "reg_lambda": [0.0, 0.5]
}

resultados_tuning = []

for params in ParameterGrid(param_grid):
    modelo_tuned = Pipeline([
        ("preprocess", preprocess),
        ("model", LGBMClassifier(
            random_state=SEED,
            class_weight="balanced",
            **params
        ))
    ])

    modelo_tuned.fit(X_train, y_train)
    p_valid_tuned = modelo_tuned.predict_proba(X_valid)[:, 1]

    resultados_tuning.append({
        **params,
        "roc_auc_valid": roc_auc_score(y_valid, p_valid_tuned),
        "pr_auc_valid": average_precision_score(y_valid, p_valid_tuned),
        "brier_valid": brier_score_loss(y_valid, p_valid_tuned)
    })

df_tuning = pd.DataFrame(resultados_tuning).sort_values(
    ["pr_auc_valid", "roc_auc_valid"],
    ascending=[False, False]
)

display(df_tuning.head(10))

[LightGBM] [Info] Number of positive: 3887, number of negative: 16604
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013719 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23216
[LightGBM] [Info] Number of data points in the train set: 20491, number of used features: 110
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number of positive: 3887, number of negative: 16604
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020944 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23216
[LightGBM] [Info] Number of data points in the train set: 20491, number of used features: 110
[LightGBM] [Info

,colsample_bytree,learning_rate,min_child_samples,n_estimators,num_leaves,reg_alpha,reg_lambda,subsample,roc_auc_valid,pr_auc_valid,brier_valid
376,1.0,0.01,50,400,63,0.0,0.0,0.7,0.648205,0.382861,0.194559
377,1.0,0.01,50,400,63,0.0,0.0,1.0,0.648205,0.382861,0.194559
310,1.0,0.01,20,200,63,0.5,0.5,0.7,0.650052,0.381092,0.202708
311,1.0,0.01,20,200,63,0.5,0.5,1.0,0.650052,0.381092,0.202708
352,1.0,0.01,50,200,63,0.0,0.0,0.7,0.652714,0.381089,0.202291
353,1.0,0.01,50,200,63,0.0,0.0,1.0,0.652714,0.381089,0.202291
354,1.0,0.01,50,200,63,0.0,0.5,0.7,0.647632,0.380958,0.203358
355,1.0,0.01,50,200,63,0.0,0.5,1.0,0.647632,0.380958,0.203358
306,1.0,0.01,20,200,63,0.0,0.5,0.7,0.654613,0.380919,0.201366
307,1.0,0.01,20,200,63,0.0,0.5,1.0,0.654613,0.380919,0.201366


In [ ]:
melhor_linha = df_tuning.sort_values(
    ["pr_auc_valid", "roc_auc_valid"],
    ascending=[False, False]
).iloc[0]

best_params = {
    "n_estimators": int(melhor_linha["n_estimators"]),
    "learning_rate": float(melhor_linha["learning_rate"]),
    "num_leaves": int(melhor_linha["num_leaves"]),
    "min_child_samples": int(melhor_linha["min_child_samples"]),
    "subsample": float(melhor_linha["subsample"]),
    "colsample_bytree": float(melhor_linha["colsample_bytree"]),
    "reg_alpha": float(melhor_linha["reg_alpha"]),
    "reg_lambda": float(melhor_linha["reg_lambda"]),
}

print("Best hyperparameters:")
print(best_params)

Best hyperparameters:
{'n_estimators': 400, 'learning_rate': 0.01, 'num_leaves': 63, 'min_child_samples': 50, 'subsample': 0.7, 'colsample_bytree': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 0.0}


## 12. Final Model

In [ ]:
final_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LGBMClassifier(
        random_state=SEED,
        class_weight="balanced",
        **best_params
    ))
])

final_model.fit(X_train, y_train)

p_valid_final = final_model.predict_proba(X_valid)[:, 1]
p_test_final = final_model.predict_proba(X_test)[:, 1]

print("FINAL VALID ROC AUC:", roc_auc_score(y_valid, p_valid_final))
print("FINAL VALID PR AUC :", average_precision_score(y_valid, p_valid_final))
print("FINAL VALID BRIER  :", brier_score_loss(y_valid, p_valid_final))

print("FINAL TEST ROC AUC:", roc_auc_score(y_test, p_test_final))
print("FINAL TEST PR AUC :", average_precision_score(y_test, p_test_final))
print("FINAL TEST BRIER  :", brier_score_loss(y_test, p_test_final))

[LightGBM] [Info] Number of positive: 3887, number of negative: 16604
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014326 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23216
[LightGBM] [Info] Number of data points in the train set: 20491, number of used features: 110
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
FINAL VALID ROC AUC: 0.6482052219192482
FINAL VALID PR AUC : 0.3828614897206802
FINAL VALID BRIER  : 0.19455854818724555
FINAL TEST ROC AUC: 0.6348125517812758
FINAL TEST PR AUC : 0.2783023163501342
FINAL TEST BRIER  : 0.1858109881064753


## 13. Business Evaluation — Who Should Be Approached?

Model quality should also be judged in terms of prioritization power.

In [ ]:
def avaliar_lift(y_true, y_score, n_bins=10):
    base = pd.DataFrame({"y": y_true, "score": y_score}).sort_values("score", ascending=False).copy()
    base["decil"] = pd.qcut(base["score"].rank(method="first"), q=n_bins, labels=False) + 1

    resumo = (
        base.groupby("decil")
            .agg(
                taxa_evento=("y", "mean"),
                volume=("y", "size")
            )
            .reset_index()
    )

    resumo["lift"] = resumo["taxa_evento"] / base["y"].mean()
    return resumo.sort_values("decil")

def metricas_topk(y_true, y_score, ks=(0.05, 0.10, 0.20)):
    base = pd.DataFrame({"y": y_true, "score": y_score}).sort_values("score", ascending=False).copy()
    total_positivos = base["y"].sum()
    saida = []

    for k in ks:
        n = max(1, int(len(base) * k))
        top = base.head(n)

        precision = top["y"].mean()
        recall = top["y"].sum() / total_positivos if total_positivos > 0 else np.nan
        lift = precision / base["y"].mean()

        saida.append({
            "top_k": k,
            "n_clientes": n,
            "precision": precision,
            "recall": recall,
            "lift": lift
        })

    return pd.DataFrame(saida)

lift_test = avaliar_lift(y_test, p_test_final, n_bins=10)
display(lift_test)

fig = px.bar(lift_test, x="decil", y="lift", title="Lift by decile — test set")
fig.show()

display(metricas_topk(y_test, p_test_final))

,decil,taxa_evento,volume,lift
0,1,0.098592,71,0.510356
1,2,0.042857,70,0.221849
2,3,0.171429,70,0.887395
3,4,0.169014,71,0.874896
4,5,0.214286,70,1.109244
5,6,0.157143,70,0.813445
6,7,0.225352,71,1.166529
7,8,0.285714,70,1.478992
8,9,0.214286,70,1.109244
9,10,0.352113,71,1.822701


,top_k,n_clientes,precision,recall,lift
0,0.05,35,0.342857,0.088235,1.774790
1,0.10,70,0.357143,0.183824,1.848739
2,0.20,140,0.285714,0.294118,1.478992


In [ ]:
test_result = test[["ID", "data"]].copy()
test_result["score_cross_sell"] = p_test_final
test_result["target_real"] = y_test.values

ranking_clientes = test_result.sort_values("score_cross_sell", ascending=False).copy()
display(ranking_clientes.head(20))

,ID,data,score_cross_sell,target_real
4357,220,2024-07-01,0.739305,0.0
5966,299,2024-08-01,0.709553,0.0
6067,304,2024-06-01,0.706412,0.0
4358,220,2024-08-01,0.678661,0.0
38509,1927,2024-06-01,0.678425,1.0
16250,813,2024-07-01,0.657701,0.0
5965,299,2024-07-01,0.652933,0.0
47799,2395,2024-06-01,0.642330,1.0
38510,1927,2024-07-01,0.635889,1.0
45584,2285,2024-06-01,0.633784,1.0


## 14. When Should Customers Be Approached?

We revisit the pre-adoption trajectory to identify a likely action window.

In [ ]:
fig = px.line(trajetoria, x="meses_para_adocao", y="uso_total_medio",
              title="Usage trajectory before adoption")
fig.show()

fig = px.line(trajetoria, x="meses_para_adocao", y="sistemas_ativos_medios",
              title="System breadth trajectory before adoption")
fig.show()

trajetoria

,meses_para_adocao,uso_total_medio,sistemas_ativos_medios
6,6,5.728603,2.667671
5,5,5.836664,2.764862
4,4,6.016230,2.742994
3,3,6.256927,2.831762
2,2,6.318907,2.822094
1,1,6.247340,2.806358
0,0,6.729558,2.957560


## 15. Why Are Customers More Likely to Adopt?

We inspect the final LightGBM model to identify the main signals driving propensity.

In [ ]:
modelo_lgb_final = final_model.named_steps["model"]
nomes_features = final_model.named_steps["preprocess"].get_feature_names_out()

feature_importance = pd.DataFrame({
    "feature": nomes_features,
    "importance": modelo_lgb_final.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(20))

fig = px.bar(
    feature_importance.head(20).sort_values("importance"),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 20 most important features"
)
fig.show()

,feature,importance
1,num__afinidade_cross_sell,2615
0,num__maturidade_latente,2281
2,num__life_time_m,662
10,num__uso_total,580
32,num__qtd_sistemas_ativos_rollstd_6,531
12,num__uso_total_lag1,516
21,num__uso_total_rollmin_6,456
94,num__uso_sistema_5_rollmax_6,426
82,num__uso_sistema_4_rollmax_6,350
106,num__uso_sistema_6_rollmax_6,348


## 16. Calibration

A well-ranked model is useful, but calibration also matters when probabilities are used operationally.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, p_test_final, n_bins=10)

df_calib = pd.DataFrame({
    "prob_predita_media": prob_pred,
    "freq_real": prob_true
})

display(df_calib)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_calib["prob_predita_media"],
    y=df_calib["freq_real"],
    mode="lines+markers",
    name="Model"
))
fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    name="Perfect calibration"
))
fig.update_layout(
    title="Calibration curve — test set",
    xaxis_title="Predicted probability",
    yaxis_title="Observed frequency"
)
fig.show()

,prob_predita_media,freq_real
0,0.087899,0.125000
1,0.155474,0.089286
2,0.252207,0.108108
3,0.348971,0.188482
4,0.443228,0.248649
5,0.549650,0.239130
6,0.629544,0.476190
7,0.718423,0.000000


## 17. Final Conclusion

This notebook shows how cross-sell can be framed as a **temporal predictive problem** and translated into a **business prioritization tool**.

By combining:

- realistic longitudinal data
- temporal feature engineering
- leakage-safe target construction
- chronological validation
- hyperparameter tuning
- business-oriented evaluation

the solution supports three practical decisions:

1. **Who** should be targeted  
2. **When** the outreach is most timely  
3. **Why** some customers are more likely to convert  

This makes the project suitable not only as a modeling exercise, but also as a portfolio-ready business case.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/Notebooks/Projetos"

'Análise de Risco de Credito Completo 2_clean.ipynb'
'Análise de Risco de Credito Completo 2.ipynb'
 Analise_de_Risco_de_Credito_Completo.ipynb
 Cross_Sell_Synthetic_Data.ipynb
 cross_sell_top1_notebook.ipynb
 cross_sell_top1_notebook_synthetic.ipynb
 requirements.txt


In [ ]:
import nbformat

input_path = "/content/drive/MyDrive/Notebooks/Projetos/Cross_Sell_Synthetic_Data.ipynb"
output_path = "/content/drive/MyDrive/Notebooks/Projetos/Cross_Sell_Synthetic_Data_clean.ipynb"

nb = nbformat.read(input_path, as_version=4)

if "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

nbformat.write(nb, output_path)

from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>